Data Ingestion : the process of collecting, importing, and loading data from different sources into a system where it can be stored, processed, or analyzed.

In [7]:
from langchain_core.documents import Document 

In [8]:
doc = Document(
    page_content = "this is the main content of RAG creation",
    metadata = {
        'author':'srina' ,
        'pages' : 1 ,
        'source' : 'rag.txt',
        'date_created': '26-02-2026'
    }
)
doc

Document(metadata={'author': 'srina', 'pages': 1, 'source': 'rag.txt', 'date_created': '26-02-2026'}, page_content='this is the main content of RAG creation')

In [9]:
import os
os.makedirs("data/text_files", exist_ok=True)

In [10]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("data/text_files/python_intro.txt", encoding="utf-8")

document = loader.load()

print(document)

[Document(metadata={'source': 'data/text_files/python_intro.txt'}, page_content='Python is a high-level, interpreted programming language that is widely used for general-purpose programming. It was created by **Guido van Rossum** and first released in **1991**. \nPython was designed with a focus on simplicity and readability, which makes it easy for beginners to learn and use. Its syntax is clear and concise, allowing developers to write programs with fewer lines of code compared to many other programming languages.\n\nOne of the main reasons for Python’s popularity is its versatility. It is used in many fields such as **web development, data science, machine learning, artificial intelligence, automation, and software development**.\n Popular frameworks like **Django and Flask** are used for web applications, while libraries such as **NumPy, Pandas, Matplotlib, and TensorFlow** support data analysis and machine learning tasks. Because of these powerful libraries, \n Python has become a

In [11]:
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader

dir_loader = DirectoryLoader(
    "data/pdfs",
    glob="*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=False
)

pdf_documents = dir_loader.load()

for doc in pdf_documents:
    print("Source:", doc.metadata["source"])
    print(doc.page_content[:200])
    print("-------------")

Source: data\pdfs\pdf.pdf
A P J Abdul Kalam Departing speech 
 
 
Friends, I am delighted to address you all, in the country and those living abroad, after 
working with you and completing five beautiful and eventful years in 
-------------
Source: data\pdfs\pdf.pdf
for it to achieve before 2020. This question reflects how the desire to live in developed 
India has entered into the minds of the youth. The same feelings are echoed by over 
fifteen lakh youth, whom
-------------
Source: data\pdfs\pdf.pdf
Let me now share with you, the enriching experience I had, while meeting more than 
6000 farmers from different States and Union Territories visiting Rashtrapati Bhavan. 
They evinced keen interest in
-------------
Source: data\pdfs\pdf.pdf
the children and the members of the village participated in the relief operation with the 
Armed Forces 
bravely and were smiling when I went to meet them. They interacted with me and said 
that the s
-------------
Source: data\pdfs\pdf.pdf
Hyderabad 

In [12]:
for doc in pdf_documents:
    print(doc.metadata)
print(len(pdf_documents))

{'producer': 'GPL Ghostscript 8.15', 'creator': 'PScript5.dll Version 5.2', 'creationdate': 'D:20070730160943', 'source': 'data\\pdfs\\pdf.pdf', 'file_path': 'data\\pdfs\\pdf.pdf', 'total_pages': 7, 'format': 'PDF 1.3', 'title': 'Microsoft Word - Document1', 'author': 'Shri', 'subject': '', 'keywords': '', 'moddate': 'D:20070730160943', 'trapped': '', 'modDate': 'D:20070730160943', 'creationDate': 'D:20070730160943', 'page': 0}
{'producer': 'GPL Ghostscript 8.15', 'creator': 'PScript5.dll Version 5.2', 'creationdate': 'D:20070730160943', 'source': 'data\\pdfs\\pdf.pdf', 'file_path': 'data\\pdfs\\pdf.pdf', 'total_pages': 7, 'format': 'PDF 1.3', 'title': 'Microsoft Word - Document1', 'author': 'Shri', 'subject': '', 'keywords': '', 'moddate': 'D:20070730160943', 'trapped': '', 'modDate': 'D:20070730160943', 'creationDate': 'D:20070730160943', 'page': 1}
{'producer': 'GPL Ghostscript 8.15', 'creator': 'PScript5.dll Version 5.2', 'creationdate': 'D:20070730160943', 'source': 'data\\pdfs\\p

In [13]:
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader

dir_loader = DirectoryLoader(
    "data/pdfs",
    glob="*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=False
)

pdf_documents = dir_loader.load()

pdf_documents

[Document(metadata={'producer': 'GPL Ghostscript 8.15', 'creator': 'PScript5.dll Version 5.2', 'creationdate': 'D:20070730160943', 'source': 'data\\pdfs\\pdf.pdf', 'file_path': 'data\\pdfs\\pdf.pdf', 'total_pages': 7, 'format': 'PDF 1.3', 'title': 'Microsoft Word - Document1', 'author': 'Shri', 'subject': '', 'keywords': '', 'moddate': 'D:20070730160943', 'trapped': '', 'modDate': 'D:20070730160943', 'creationDate': 'D:20070730160943', 'page': 0}, page_content='A P J Abdul Kalam Departing speech \n \n \nFriends, I am delighted to address you all, in the country and those living abroad, after \nworking with you and completing five beautiful and eventful years in Rashtrapati \nBhavan. Today, it is indeed a thanks giving occasion. I would like to narrate, how I \nenjoyed every minute of my tenure enriched by the wonderful association from each one \nof you, hailing from different walks of life, be it politics, science and technology, \nacademics, arts, literature, business, judiciary, adm

In [14]:
type(pdf_documents[0])

langchain_core.documents.base.Document

embedding and vectorstoreDB

In [15]:
 import numpy as np
 from sentence_transformers import SentenceTransformer
 import chromadb
 from chromadb.config import Settings
 import uuid
 from typing import List, Dict, Any , Tuple
 from sklearn.metrics.pairwise import cosine_similarity


In [16]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model=None
        self._load_model()
#A protected function is a function (method) in object-oriented programming that can be accessed only within the 
# same class and its derived (child) classes, but not from outside the class.starts with _
    def _load_model(self):
        """loading the sentence transformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading the model {self.model_name}: {e}")
            raise 
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:

        """generate embeddings for a list of texts """
        if not self.model:
            raise ValueError("Model not found")
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    '''
    def get_embedding_dimension(self) -> int:
        """get the embedding dimension of the model"""
        if not self.model:
            raise ValueError("Model not found")
        return self.model.get_sentence_embedding_dimension()
    '''
embedding_manager = EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 534.36it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


Vectorstore

In [17]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 61


In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(pdf_documents)

print(len(chunks))

61


In [19]:
chunks

[Document(metadata={'producer': 'GPL Ghostscript 8.15', 'creator': 'PScript5.dll Version 5.2', 'creationdate': 'D:20070730160943', 'source': 'data\\pdfs\\pdf.pdf', 'file_path': 'data\\pdfs\\pdf.pdf', 'total_pages': 7, 'format': 'PDF 1.3', 'title': 'Microsoft Word - Document1', 'author': 'Shri', 'subject': '', 'keywords': '', 'moddate': 'D:20070730160943', 'trapped': '', 'modDate': 'D:20070730160943', 'creationDate': 'D:20070730160943', 'page': 0}, page_content='A P J Abdul Kalam Departing speech \n \n \nFriends, I am delighted to address you all, in the country and those living abroad, after \nworking with you and completing five beautiful and eventful years in Rashtrapati \nBhavan. Today, it is indeed a thanks giving occasion. I would like to narrate, how I \nenjoyed every minute of my tenure enriched by the wonderful association from each one \nof you, hailing from different walks of life, be it politics, science and technology,'),
 Document(metadata={'producer': 'GPL Ghostscript 8.1

In [20]:
##convert the text to embeddings
texts=[doc.page_content for doc in chunks]
texts

#generate the embeddings 
embeddings = embedding_manager.generate_embeddings(texts)

##store in vectore datavase 

vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 61 texts...


Batches: 100%|██████████| 2/2 [00:00<00:00,  2.52it/s]


Generated embeddings with shape: (61, 384)
Adding 61 documents to vector store...
Successfully added 61 documents to vector store
Total documents in collection: 122


RAG RETRIEVER PIPELINE FROM VECTORSTORE

In [21]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [22]:
rag_retriever

In [23]:
rag_retriever.retrieve("What is Dr. Kalam’s dream?")

Retrieving documents for query: 'What is Dr. Kalam’s dream?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 30.81it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_356ba12d_58',
  'content': 'efficient, clean and affordable. \nI am happy that Dr. Pillai has come out with the book ‘40 Years with Abdul \nKalam’ providing a gripping, first-hand account of Dr. Kalam. \nThe life of Dr. Kalam sends a powerful message about how difficulties and \nsetbacks when taken in the right spirit, serve as the key ingredients in making us \nrobust in character and mindset. Today’s generation should draw inspiration from \nhim and dedicatedly work towards building a strong, self-reliant and inclusive',
  'metadata': {'keywords': '',
   'subject': '',
   'title': 'Microsoft Word - Document1',
   'creationDate': "D:20201203202843+05'30'",
   'author': 'Admin',
   'moddate': '2020-12-03T20:28:43+05:30',
   'trapped': '',
   'content_length': 492,
   'total_pages': 3,
   'page': 2,
   'modDate': "D:20201203202843+05'30'",
   'creator': 'PScript5.dll Version 5.2',
   'producer': 'Acrobat Distiller 7.0 (Windows)',
   'creationdate': '2020-12-03T20:28:43+05:3

VECTORDB CONTEXT PIPELINE WIRH LLM OUTPUT

In [ ]:
from langchain_groq import ChatGroq

import os
from dotenv import load_dotenv
load_dotenv()
groq_api_key="your_api_key"
llm=ChatGroq(groq_api_key=groq_api_key, model_name = "llama-3.1-8b-instant", temperature=0.2, max_tokens=1024)

def rag_simple(query, retriever, llm, top_k=3):
    # retrieve context
    results = retriever.retrieve(query, top_k=top_k)

    context = "\n\n".join([doc["content"] for doc in results]) if results else ""

    if not context:
        return "No relevant context found to answer the question."

# generate the answer using GROQ LLM
    prompt = f"""Use the following context to answer the question concisely.

        Context:
        {context}

        Question: {query}

        Answer:"""

    response = llm.invoke(prompt.format(context=context, query=query))

    return response.content



In [30]:
answer=rag_simple("what is abdul kalam speech?, Explain", rag_retriever,llm)
print(answer)

Retrieving documents for query: 'what is abdul kalam speech?, Explain'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 32.23it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Abdul Kalam's speech is his departing speech as the President of India, where he expresses gratitude to the nation and its people for the opportunity to serve as the President for five years.


In [ ]:
enhanced RAG pipeline Features

In [31]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("abdul kalam dream and his missions", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'abdul kalam dream and his missions'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 30.41it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: Dr. Abdul Kalam's dream was to make India a Developed Nation. His missions included promoting friendship and knowledge globally, and inspiring the youth to work towards building a strong, self-reliant, and inclusive India.
Sources: [{'source': 'data\\pdfs\\pdf2.pdf', 'page': 0, 'score': 0.2617446780204773, 'preview': 'children, who are the most precious asset. From there he has been transmitting the \ndream of how to make India a Developed Nation”. \nAs President, Dr Kalam personified dignity and optimism throughout India and \nabroad. He was regarded in every country as a strong promoter of friendship and \nknowledg...'}, {'source': 'data\\pdfs\\pdf2.pdf', 'page': 0, 'score': 0.2617446780204773, 'preview': 'children, who are the most precious asset. From there he has been transmitting the \ndream of how to make India a Developed Nation”. \nAs President, Dr Kalam personified dignity and optimism throughout India and \nabroad. He was regarded in every country as a strong promote

In [ ]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("Dr. Abdul Kalams dream and passion towards development of country", top_k=3, min_score=0.3, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'abdul kalams dream and passion towards development of country'
Top K: 3, Score threshold: 0.3
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 26.36it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
children, who are the most precious asset. From there he has been transmitting the 
dream of how to make India a Developed Nation”. 
As President, Dr Kalam personified dig

nity and optimism throughout India and 
abroad. He was regarded in every country as a strong promoter of friendship and 
knowledge. In recognition of his contributions, NASA had named a newly 
discovered organism found on the International Space Station after Dr. Kalam in 
2017.

children, who are the most precious asset. From there he has been transmitting the 
dream of how to make India a Developed Nation”. 
As President, Dr Kalam personified dignity and optimism throughout India and 
abroad. He was regarded in every country as a strong promoter of friendship and 
knowledge. In recognition of his contributions, NASA had named a newly 
discovered organism found on the International Space Station after Dr. Kalam in 
2017.

capability to become a developed nation in the near future. 
In fact, Dr. Kalam generated Vision for each State in the form of Display Charts. 
He invited the MPs to the Rashtrapati Bhavan and explained to them the value, 
core competence and development schemes of t